# part 1

In [1]:
from utils import load_data, train, compute_uas, compute_uas_attn

In [2]:
train_sents, test_sents = load_data()
print(f'Train: {len(train_sents)} sentences, Test: {len(test_sents)} sentences')

Train: 3522 sentences, Test: 392 sentences


In [3]:
avg_weights, raw_weights = train(train_sents)
print('Training done.')

Training done.


In [4]:
uas_avg = compute_uas(test_sents, avg_weights)
print(f'UAS (averaged): {uas_avg:.4f}')

UAS (averaged): 0.1768


In [5]:
uas_raw = compute_uas(test_sents, raw_weights)
print(f'UAS (raw final): {uas_raw:.4f}')

UAS (raw final): 0.1676


we initially thought the low avareged UAS is a result of underfitting in only 2 itterations but in fact we see the latter itteration is already overfitting, so it is simply a limitted model on a limitted dataset which causes this lousy performance

# part 2

In [6]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model.eval()
print('BERT loaded.')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT loaded.


In [7]:
uas_layer0 = compute_uas_attn(test_sents, tokenizer, model, layer=0)
print(f'UAS BERT layer=0:  {uas_layer0:.4f}')

UAS BERT layer=0:  0.2304


In [8]:
uas_layer5 = compute_uas_attn(test_sents, tokenizer, model, layer=5)
print(f'UAS BERT layer=5:  {uas_layer5:.4f}')

UAS BERT layer=5:  0.1377


In [9]:
uas_layer11 = compute_uas_attn(test_sents, tokenizer, model, layer=11)
print(f'UAS BERT layer=11: {uas_layer11:.4f}')

UAS BERT layer=11: 0.1201


ok maybe these low numbers are expected? anyhow we see that out of all the early layers capture it best, so it is syntax that best explain our parsing more than semantic information.

 ## Part 2.3 — Comparison
 
  | Parser | UAS |
  |---|---|
  | Perceptron MST (averaged weights) | 0.1765 |
  | BERT attention, layer=0 | 0.2304 |
  | BERT attention, layer=5 | 0.1377 |
  | BERT attention, layer=11 | 0.1201 |

  The perceptron MST parser requires labeled training data and explicit feature engineering, while the BERT-based parser requires no task-specific training at all — it uses attention weights extracted from a model pre-trained on language
  modeling. In terms of inference cost, the perceptron is much cheaper: scoring edges is a simple dot product, whereas BERT requires a full forward pass through a large transformer for each sentence. Accuracy-wise, the best BERT setting
  (layer=0) slightly outperforms the perceptron here, which is notable given that BERT was never trained for parsing. The fact that attention weights correlate with syntactic structure at all likely reflects that understanding language at a
  surface level requires tracking local grammatical relationships — even a model trained only to predict masked words must implicitly learn which words govern which others to do so accurately.                               

  Paste this as a new markdown cell at the bottom. You'll want to update the UAS numbers once you have final results.

